<a href="https://colab.research.google.com/github/edgardlt03/ICO-Trabajos/blob/main/Vector_Stores_y_B%C3%BAsqueda_Sem%C3%A1ntica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers scikit-learn kagglehub --quiet


In [2]:
from sentence_transformers import SentenceTransformer

### Carga del Animal Fun Facts Dataset

Fuente: https://github.com/ekohrt/animal-fun-facts-dataset


In [25]:
import csv, io, urllib.request
RAW_URL = (
    "https://raw.githubusercontent.com/ekohrt/"
    "animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
)

with urllib.request.urlopen(RAW_URL) as resp:
    csv_text = resp.read().decode("utf-8")

animal_docs: list[Document] = []
for row in csv.DictReader(io.StringIO(csv_text)):
    text = row.get("text", "").strip()
    if text:
        animal_docs.append(Document(
            text=text,
            metadata={
                "animal_name":    row.get("animal_name", "").strip(),
                "source":         row.get("source", "").strip(),
                "media_link":     row.get("media_link", "").strip(),
                "wikipedia_link": row.get("wikipedia_link", "").strip(),
            }
        ))

print(f"Documentos cargados: {len(animal_docs):,}")
print("Metadata:", animal_docs[50].metadata)


Documentos cargados: 7,731
Metadata: {'animal_name': 'bat', 'source': 'https://www.animalfactsencyclopedia.com/Bat-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Bat'}


## Parte I: VectorStore básico

In [48]:
import numpy as np

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        q_emb = self.model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
        scores = cosine_similarity(q_emb, self.embeddings)[0]
        top_indices = np.argsort(scores)[::-1][:min(top_k, len(self.documents))]
        return [SearchResult(float(scores[i]), self.documents[i]) for i in top_indices]


### Instancia de VectorStore y carga de documentos

In [27]:
model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dimensiones

vs = VectorStore(model)
vs.add_documents(animal_docs)

print(f"Documentos : {len(vs.documents):,}")
print(f"Embeddings : {vs.embeddings.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Documentos : 7,731
Embeddings : (7731, 384)


### 5 consultas de ejemplo score, texto y metadatos


In [54]:
def show(results: list[SearchResult]):

    for i, r in enumerate(results, 1):
        print(f"{i}.Score       : {r.score:.4f}")
        print(f"  Texto       : {r.document.text[:120]}")
        print(f"  animal_name : {r.document.metadata['animal_name']}")
        print(f"  source      : {r.document.metadata['source']}")
        print(f"  wikipedia   : {r.document.metadata['wikipedia_link']}")
        print(f"  media_link  : {r.document.metadata['media_link']}\n")



In [63]:
print("CONSULTA 1\n")
show(vs.search("animals that sleep for many hours a day", top_k=3))
print("\nCONSULTA 2\n")
show(vs.search("dolphin communication and intelligence", top_k=3))
print("\nCONSULTA 3\n")
show(vs.search("poisonous animals that can kill humans", top_k=3))
print("\nCONSULTA 4\n")
show(vs.search("birds that migrate thousands of miles", top_k=3))
print("\nCONSULTA 5\n")
show(vs.search("fastest animal on land", top_k=3))

CONSULTA 1

1.Score       : 0.7291
  Texto       : They often sleep 16 hours a day!.
In addition to being solitary animals, armadillos also like to sleep—a lot.
  animal_name : armadillo
  source      : https://factanimal.com/armadillo/
  wikipedia   : /wiki/Armadillo
  media_link  : 

2.Score       : 0.7173
  Texto       : These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of
  animal_name : coatimundi
  source      : https://seaworld.org/animals/facts/mammals/coatimundi/
  wikipedia   : /wiki/Coati
  media_link  : 

3.Score       : 0.7156
  Texto       : Anteaters sleep as much as 15 hours each day.
  animal_name : giant anteater
  source      : https://seaworld.org/animals/facts/mammals/giant-anteater/
  wikipedia   : /wiki/Giant_anteater
  media_link  : 


CONSULTA 2

1.Score       : 0.6805
  Texto       : Dolphins communicate with clicks and whistles..
This helps them navigate, warn of potential predators and hunt 

### Segundo dataset: Rotten Tomatoes Movie Reviews

Fuente: https://www.kaggle.com/datasets/subhajournal/movie-rating




In [64]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "subhajournal/movie-rating",
    "Rotten Tomatoes Movies.csv",
)

print(f"Filas totales: {len(df):,}")
print("Columnas:", df.columns.tolist())


/tmp/ipykernel_32666/2361529068.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 21.6M/21.6M [00:00<00:00, 30.0MB/s]


Filas totales: 16,638
Columnas: ['movie_title', 'movie_info', 'critics_consensus', 'rating', 'genre', 'directors', 'writers', 'cast', 'in_theaters_date', 'on_streaming_date', 'runtime_in_minutes', 'studio_name', 'tomatometer_status', 'tomatometer_rating', 'tomatometer_count', 'audience_rating', 'audience_count']


In [67]:
movie_docs: list[Document] = []
for _, row in df.iterrows():
    text = str(row.get("critics_consensus", "")).strip()
    if text and text != "nan":
        movie_docs.append(Document(
            text=text,
            metadata={
                "movie_title":        str(row.get("movie_title", "")).strip(),
                "genre":              str(row.get("genre", "")).strip(),
                "rating":             str(row.get("rating", "")).strip(),
                "tomatometer_status": str(row.get("tomatometer_status", "")).strip(),
            }
        ))

print(f"Documentos cargados: {len(movie_docs):,}")
print("Ejemplo:", movie_docs[0].text[:100])
print("Metadata:", movie_docs[0].metadata)


Documentos cargados: 8,309
Ejemplo: Though it may seem like just another Harry Potter knockoff, Percy Jackson benefits from a strong sup
Metadata: {'movie_title': 'Percy Jackson & the Olympians: The Lightning Thief', 'genre': 'Action & Adventure, Comedy, Drama, Science Fiction & Fantasy', 'rating': 'PG', 'tomatometer_status': 'Rotten'}


##Parte II: Filtering by metadata

In [69]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embs = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
        self.embeddings = new_embs if self.embeddings is None else np.vstack([self.embeddings, new_embs])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if not self.documents:
            return []

        # Si hay filtros selecciona solo los docs que los cumplan
        if metadata_filter:
            valid = [
                i for i, doc in enumerate(self.documents)
                if all(doc.metadata.get(k, "").lower() == v.lower()
                       for k, v in metadata_filter.items())]
        else:
            valid = list(range(len(self.documents)))

        if not valid:
            return []

        q_emb = self.model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
        scores = cosine_similarity(q_emb, self.embeddings[valid])[0]
        top_local = np.argsort(scores)[::-1][:min(top_k, len(valid))]

        return [SearchResult(float(scores[j]), self.documents[valid[j]]) for j in top_local]


### Instancia de FilteredVectorStore

In [70]:
fvs = FilteredVectorStore(model)
fvs.add_documents(movie_docs)

print(f"Documentos : {len(fvs.documents):,}")
print(f"Embeddings : {fvs.embeddings.shape}")


Batches:   0%|          | 0/260 [00:00<?, ?it/s]

Documentos : 8,309
Embeddings : (8309, 384)


### 5 consultas con filtro de metadata score, texto y metadatos

In [75]:
def show_movies(results: list[SearchResult], metadata_filter: dict | None = None):
    filtro = str(metadata_filter) if metadata_filter else "(sin filtro)"

    print(f"Filtro : {filtro}")

    for i, r in enumerate(results, 1):
        print(f"{i}.Score              : {r.score:.4f}")
        print(f"    Texto              : {r.document.text[:110]}...")
        print(f"    movie_title        : {r.document.metadata['movie_title']}")
        print(f"    genre              : {r.document.metadata['genre']}")
        print(f"    rating             : {r.document.metadata['rating']}")
        print(f"    tomatometer_status : {r.document.metadata['tomatometer_status']}")


In [80]:
print("Query 1 horror solo Certified Fresh\n")
f1 = {"tomatometer_status": "Certified Fresh"}
show_movies(fvs.search("scary horror movie with terrifying atmosphere", top_k=3, metadata_filter=f1), f1)

print("\nQuery 2 comedia romántica solo rating PG\n")
f2 = {"rating": "PG"}
show_movies(fvs.search("romantic comedy funny and heartwarming love story", top_k=3, metadata_filter=f2), f2)

print("\nQuery 3 acción solo Rotten \n")
f3 = {"tomatometer_status": "Rotten"}
show_movies(fvs.search("action packed movie with great stunts and explosions", top_k=3, metadata_filter=f3), f3)

print("\nQuery 4 drama solo rating R\n")
f4 = {"rating": "R"}
show_movies(fvs.search("emotional drama with powerful performances", top_k=3, metadata_filter=f4), f4)

print("\nQuery 5 animación solo Fresh\n")
f5 = {"tomatometer_status": "Fresh"}
show_movies(fvs.search("animated movie for kids with colorful visuals and fun story", top_k=3, metadata_filter=f5), f5)


Query 1 horror solo Certified Fresh

Filtro : {'tomatometer_status': 'Certified Fresh'}
1.Score              : 0.6315
    Texto              : This classic low budget horror film combines just the right amount of gore and black humor, giving The Evil De...
    movie_title        : The Evil Dead
    genre              : Horror, Science Fiction & Fantasy
    rating             : R
    tomatometer_status : Certified Fresh
2.Score              : 0.6260
    Texto              : Deeply unnerving and surprisingly poignant, The Orphanage is an atmospheric, beautifully crafted haunted house...
    movie_title        : The Orphanage
    genre              : Drama, Horror, Mystery & Suspense
    rating             : R
    tomatometer_status : Certified Fresh
3.Score              : 0.5957
    Texto              : The Exorcist rides its supernatural theme to magical effect, with remarkable special effects and an eerie atmo...
    movie_title        : The Exorcist
    genre              : Classics, 

## Reflexión personal

En esta actividad entendí de forma más clara cómo funcionan los embeddings mque básicamente convierten el texto en números pero lo interesante es que no solo toman en cuenta las palabras exactas, sino también el significado general. También vi que filtrar con metadatos antes de buscar similitud sirve para hacer la búsqueda más específica sin tener que cambiar el modelo además noté que mientras más documentos se agregan más pesada se vuelve la búsqueda porque el sistema tiene que comparar contra todos los documentos guardados.